# Cellpose masks from a high-resolution still

This notebook uses the reusable functions in `suite2p.still_cellpose`. Inspect the alignment overlay before extracting traces.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from suite2p.still_cellpose import (
    extract_with_predefined_stats,
    load_cellpose_masks,
    load_still_channel,
    load_suite2p_context,
    masks_to_suite2p_stats,
    plot_corrected_traces,
    plot_mask_overlay,
    resize_label_masks,
    run_cellpose_on_still,
    shift_label_masks,
)

In [ ]:
# Paths
PLANE0 = Path(r"C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-4_Day5\suite2p\plane0")
STILL_PATH = Path(r"C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating\BP\1-4_Day5\snap\1-4_Day5_snap.tif")
MASK_PATH = None  # Set to an existing Cellpose .npy to skip inference
OUTPUT_PATH = PLANE0.parent / "plane0_still_mask_test"

# Still and Cellpose settings
STILL_CHANNEL = 1       # zero-based: second channel
STILL_CHANNEL_AXIS = 0  # this still is C,Y,X
CELLPOSE_MODEL = "cpdino"
STILL_DIAMETER = None
CELLPROB_THRESHOLD = 0.0
FLOW_THRESHOLD = 0.4

# Alignment correction at Suite2p resolution
DY, DX = 1, -1  # positive is down/right

In [ ]:
db, settings, reference_image = load_suite2p_context(
    PLANE0, image_file="detect_outputs.npy", image_key="Vcorr"
)
still = load_still_channel(
    STILL_PATH, channel=STILL_CHANNEL, channel_axis=STILL_CHANNEL_AXIS
)
print("Still shape:", still.shape)
print("Suite2p shape:", reference_image.shape)
print("Resolution ratios:", tuple(s / r for s, r in zip(still.shape, reference_image.shape)))

In [ ]:
if MASK_PATH is None:
    masks_hires = run_cellpose_on_still(
        still,
        model_name_or_path=CELLPOSE_MODEL,
        diameter=STILL_DIAMETER,
        cellprob_threshold=CELLPROB_THRESHOLD,
        flow_threshold=FLOW_THRESHOLD,
    )
else:
    masks_hires = load_cellpose_masks(MASK_PATH)
print(f"Cellpose masks: {masks_hires.max()}")

In [ ]:
masks = resize_label_masks(masks_hires, reference_image.shape)
masks = shift_label_masks(masks, dy=DY, dx=DX)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
plot_mask_overlay(still, masks_hires, "High-resolution Cellpose masks", ax=axes[0], linewidth=0.4)
plot_mask_overlay(reference_image, masks, f"Suite2p overlay (DY={DY}, DX={DX})", ax=axes[1])
plt.tight_layout()
print(f"Masks after resize/shift: {masks.max()}")

## Extraction
Continue only if the overlay above is aligned. Results are written to `OUTPUT_PATH`; the original `plane0` is not modified.

In [ ]:
stat = masks_to_suite2p_stats(masks, settings, do_soma_crop=False)
print(f"ROIs after Suite2p size/overlap checks: {len(stat)}")

outputs = extract_with_predefined_stats(
    plane_path=PLANE0,
    output_path=OUTPUT_PATH,
    stat=stat,
    db=db,
    settings=settings,
)
print("Finished:", OUTPUT_PATH)
print(sorted(path.name for path in OUTPUT_PATH.glob("*.npy")))

In [ ]:
plot_corrected_traces(OUTPUT_PATH, settings, n_traces=10, seed=0)
plt.tight_layout()